# Image Render Test — which approach works in your Jupyter?

Paste one real internal `img_url` below and run the cells. Three approaches are tried, each
clearly labelled. **Report back which one(s) render the image** and I'll wire that into the
review notebook.

- **A — direct `<img src=url>`**: simplest. Works only if Jupyter's browser can reach the URL
  with no auth (same network/VPN, no login wall).
- **B — authenticated fetch → base64 embed**: uses your Langfuse `httpx` client (verify=False,
  trust_env=False) to download the bytes and inline them. Works if the image needs the same
  auth/host as your API and direct loading fails.
- **C — plain `httpx` fetch (no auth) → base64 embed**: downloads bytes without credentials and
  inlines them. Works if the URL is fetchable from the *kernel* (server-side) but the browser
  can't reach it directly (common when the kernel is inside the network but your browser isn't).

## 0. Config — paste one real image URL + your Langfuse creds (for approach B)

In [ ]:
TEST_IMG_URL = "https://your-internal-host/path/to/circular_page.png"   # <-- paste a real img_url

# only needed for Approach B (authenticated fetch). Reuse your explorer creds:
LANGFUSE_HOST       = "https://your-langfuse.internal"
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

# display size for rendered images (px). Big enough to read a circular page:
IMG_WIDTH = 900

import httpx, base64, mimetypes
from IPython.display import display, HTML, Image
print("Config set. URL:", TEST_IMG_URL)

## Approach A — direct `<img src=url>` (no download)
If this shows the image, it's the simplest path: the browser reaches the URL directly.

In [ ]:
print("APPROACH A: direct <img src> tag\n")
display(HTML(
    f"<div style='border:2px solid #2F5597;padding:8px'>"
    f"<b>Approach A</b><br>"
    f"<img src='{TEST_IMG_URL}' style='max-width:{IMG_WIDTH}px;width:100%;height:auto;"
    f"border:1px solid #ccc' "
    f"onerror=\"this.insertAdjacentHTML('afterend','<p style=color:red>A FAILED: browser could not load this URL directly (auth or unreachable).</p>')\">"
    f"</div>"
))
print("If you see the image above -> Approach A works.")

## Approach B — authenticated fetch (Langfuse httpx) → base64 embed
Downloads via your API client (handles self-signed certs + auth), then inlines the bytes.

In [ ]:
print("APPROACH B: authenticated httpx fetch -> base64\n")
try:
    _auth_client = httpx.Client(auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
                                verify=False, trust_env=False,
                                timeout=httpx.Timeout(30.0, read=60.0), follow_redirects=True)
    r = _auth_client.get(TEST_IMG_URL)
    print("HTTP status:", r.status_code, "| content-type:", r.headers.get("content-type"),
          "| bytes:", len(r.content))
    r.raise_for_status()
    ctype = r.headers.get("content-type", "").split(";")[0] or             mimetypes.guess_type(TEST_IMG_URL)[0] or "image/png"
    b64 = base64.b64encode(r.content).decode()
    display(HTML(
        f"<div style='border:2px solid #2F5597;padding:8px'><b>Approach B</b><br>"
        f"<img src='data:{ctype};base64,{b64}' "
        f"style='max-width:{IMG_WIDTH}px;width:100%;height:auto;border:1px solid #ccc'></div>"
    ))
    print("If you see the image above -> Approach B works.")
except Exception as e:
    print("B FAILED:", type(e).__name__, e)

## Approach C — plain fetch (no auth) → base64 embed
Same as B but with no credentials — for when the kernel can reach the URL but the browser can't,
and the image needs no auth.

In [ ]:
print("APPROACH C: plain httpx fetch (no auth) -> base64\n")
try:
    _plain = httpx.Client(verify=False, trust_env=False,
                          timeout=httpx.Timeout(30.0, read=60.0), follow_redirects=True)
    r = _plain.get(TEST_IMG_URL)
    print("HTTP status:", r.status_code, "| content-type:", r.headers.get("content-type"),
          "| bytes:", len(r.content))
    r.raise_for_status()
    ctype = r.headers.get("content-type", "").split(";")[0] or             mimetypes.guess_type(TEST_IMG_URL)[0] or "image/png"
    b64 = base64.b64encode(r.content).decode()
    display(HTML(
        f"<div style='border:2px solid #2F5597;padding:8px'><b>Approach C</b><br>"
        f"<img src='data:{ctype};base64,{b64}' "
        f"style='max-width:{IMG_WIDTH}px;width:100%;height:auto;border:1px solid #ccc'></div>"
    ))
    print("If you see the image above -> Approach C works.")
except Exception as e:
    print("C FAILED:", type(e).__name__, e)

---
### How to report back
Tell me **which approach(es) rendered the image** (A, B, C, or combinations) and paste the
printed **HTTP status / content-type / bytes** lines from B and C. That tells me:
- If **A works** → review UI uses simple `<img>` tags (lightest, no download cost).
- If **A fails but B/C work** → review UI fetches bytes and embeds base64 (I'll use whichever
  of B/C succeeded — B if it needs auth, C if not).
- If **all fail** → the status/content-type lines tell us why (404, 401/403 auth, cert, wrong
  content-type) and I'll adjust.